# Download Raw Data

This notebook downloads all required raw input files into `data/raw` and merges OSM extracts into `data/raw/osm/merged/northern-germany.osm.pbf`.

Files handled:
- `population_grid.csv` (Destatis)
- `hvv_Rohdaten_GTFS_Fpl_26.ZIP` (HVV GTFS)
- `hamburg-260718.osm.pbf`
- `niedersachsen-260718.osm.pbf`
- `schleswig-holstein-260718.osm.pbf`
- merged `northern-germany.osm.pbf`

In [ ]:
from __future__ import annotations

import io
import re
import shutil
import subprocess
import zipfile
from pathlib import Path
from urllib.parse import urljoin

import requests

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
OSM_DIR = RAW_DIR / 'osm'
MERGED_DIR = OSM_DIR / 'merged'

RAW_DIR.mkdir(parents=True, exist_ok=True)
OSM_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)

In [ ]:
def download_file(url: str, destination: Path, chunk_size: int = 1024 * 1024) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get('Content-Length', '0'))
        written = 0
        with destination.open('wb') as f:
            for chunk in response.iter_content(chunk_size=chunk_size):
                if not chunk:
                    continue
                f.write(chunk)
                written += len(chunk)
                if total:
                    pct = (written / total) * 100
                    print(f'\r{destination.name}: {pct:5.1f}% ({written:,}/{total:,} bytes)', end='')
        if total:
            print()
    print(f'Saved {destination}')

## 1) Download population grid (Destatis)

Downloads the official ZIP and extracts `Zensus2022_Bevoelkerungszahl_100m-Gitter.csv` to `data/raw/population_grid.csv`.

In [ ]:
destatis_zip_url = 'https://www.destatis.de/static/DE/zensus/gitterdaten/Zensus2022_Bevoelkerungszahl.zip'
destatis_csv_name = 'Zensus2022_Bevoelkerungszahl_100m-Gitter.csv'
population_grid_path = RAW_DIR / 'population_grid.csv'

resp = requests.get(destatis_zip_url, timeout=120)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as archive:
    with archive.open(destatis_csv_name) as src, population_grid_path.open('wb') as dst:
        shutil.copyfileobj(src, dst)

print('Saved', population_grid_path)

## 2) Download HVV GTFS ZIP

Attempts to discover the direct file URL from the Hamburg transparency dataset page and saves it as `data/raw/hvv_Rohdaten_GTFS_Fpl_26.ZIP`.

In [ ]:
hvv_dataset_page = 'https://suche.transparenz.hamburg.de/dataset/hvv-fahrplandaten-gtfs-april-2026-bis-dezember-2026'
hvv_target_path = RAW_DIR / 'hvv_Rohdaten_GTFS_Fpl_26.ZIP'

page = requests.get(hvv_dataset_page, timeout=120)
page.raise_for_status()
html = page.text

hrefs = re.findall(r'href=["]([^"]+)["]', html, flags=re.IGNORECASE)
zip_candidates = []
for href in hrefs:
    lower = href.lower()
    if 'gtfs' in lower and lower.endswith('.zip'):
        zip_candidates.append(urljoin(hvv_dataset_page, href))

if not zip_candidates:
    raise RuntimeError('Could not auto-discover GTFS ZIP URL from HVV dataset page.')

hvv_download_url = zip_candidates[0]
print('Using GTFS URL:', hvv_download_url)
download_file(hvv_download_url, hvv_target_path)

## 3) Download OSM PBF extracts

Downloads the three required Geofabrik extracts into `data/raw/osm/`.

In [ ]:
osm_urls = {
    'hamburg-260718.osm.pbf': 'https://download.geofabrik.de/europe/germany/hamburg-260718.osm.pbf',
    'niedersachsen-260718.osm.pbf': 'https://download.geofabrik.de/europe/germany/niedersachsen-260718.osm.pbf',
    'schleswig-holstein-260718.osm.pbf': 'https://download.geofabrik.de/europe/germany/schleswig-holstein-260718.osm.pbf',
}

for filename, url in osm_urls.items():
    download_file(url, OSM_DIR / filename)

## 4) Merge OSM PBF files

Requires `osmium` CLI. On Ubuntu/Debian:

```bash
sudo apt install -y osmium-tool
```

Merges the three regional extracts into `data/raw/osm/merged/northern-germany.osm.pbf`.

In [ ]:
osmium_path = shutil.which('osmium')
if osmium_path is None:
    raise RuntimeError('osmium CLI is not installed. Install with: sudo apt install -y osmium-tool')

merged_output = MERGED_DIR / 'northern-germany.osm.pbf'
inputs = [
    OSM_DIR / 'hamburg-260718.osm.pbf',
    OSM_DIR / 'niedersachsen-260718.osm.pbf',
    OSM_DIR / 'schleswig-holstein-260718.osm.pbf',
]

missing = [str(p) for p in inputs if not p.exists()]
if missing:
    raise FileNotFoundError('Missing OSM input files: ' + ', '.join(missing))

cmd = [
    osmium_path,
    'merge',
    str(inputs[0]),
    str(inputs[1]),
    str(inputs[2]),
    '-o',
    str(merged_output),
    '--overwrite',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)
print('Saved', merged_output)